In [1]:
from itertools import permutations

# 计算两个字符串的最大 overlap（a 的后缀与 b 的前缀）
def overlap(a, b):
    max_olen = 0
    min_len = min(len(a), len(b))
    for i in range(1, min_len + 1):
        if a[-i:] == b[:i]:
            max_olen = i
    return max_olen

# 把一个排列中的字符串依次合并
def merge_order(strings):
    s = strings[0]
    for i in range(1, len(strings)):
        olen = overlap(s, strings[i])
        s = s + strings[i][olen:]
    return s

# 暴力 SCS：尝试所有排列，取最短的
def scs(strings):
    best = None
    for perm in permutations(strings):
        merged = merge_order(perm)
        if best is None or len(merged) < len(best):
            best = merged
    return best

In [2]:
def scs_list(strings):
    shortest = None
    results = []
    for perm in permutations(strings):
        merged = merge_order(perm)
        if shortest is None or len(merged) < shortest:
            shortest = len(merged)
            results = [merged]
        elif len(merged) == shortest:
            results.append(merged)
    return sorted(set(results))  

In [3]:
strings = ['CCT', 'CTT', 'TGC', 'TGG', 'GAT', 'ATT']

In [4]:
len(scs(strings))

11

In [5]:
len(strings)

6

In [6]:
len(scs_list(strings))

4

In [7]:
def readFastq(filename):
    sequences = []
    qualities = []
    with open(filename) as fh:
        while True:
            fh.readline()  # skip name line
            seq = fh.readline().rstrip()  # read base sequence
            fh.readline()  # skip placeholder line
            qual = fh.readline().rstrip() # base quality line
            if len(seq) == 0:
                break
            sequences.append(seq)
            qualities.append(qual)
    return sequences, qualities

In [8]:
fqresult = readFastq("ads1_week4_reads.fq")

In [9]:
sequences = fqresult[0]
sequences[:10]

['GTCCAGCAGAGCAAGTGATGCGAGAGCTGCCCATCCTCCAACCAGCATGCCCCTAGACATTGACACTGCATCGGAGTCAGGCCAAGATCCGCAGGACAGT',
 'GGAGTACGACTTCAGAGATCTCACTTGGTGTATCAACCCGCCAGAGAGAATCAAATTGGATTATGATCAATACTGTGCAGATGTGGCTGCTGAAGAACTC',
 'GCAAATTTTGATCTCTCTTGGCTTCACAATCAATTCAACCATGACCCGAGATGTAGTCATACCCCTCCTCACAAACAACGATCTCTTAATAAGGATGGCA',
 'GAGTTAATTGAAGCCCTAGATTACATTTTCATAACTGATGACATACATCTGACAGGGGAGATTTTCTCATTTTTCAGAAGTTTCGGCCACCCCAGACTTG',
 'AATGACAGAGACCGCTATGACCATTGATGCTAGGTATGCAGAACTTCTAGGAAGAGTCAGATACATGTGGAAACTGATAGATGGTTTCTTCCCTGCACTC',
 'TACGATTTCGACAAGTCGGCATGGGACATCAAAGGGTCGATCGCTCCGATACAACCTACCACCTACAGTGATGGCAGGCTGGTGCCCCAGGTCAGAGTCA',
 'AGAGGAGGCAGGCAGTTCGGGTCTCAGCAAACCATGCCTCTCAGCAATTGGATCAACTGAAGGCGGTGCACCTCGCATCCGCGGTCAGGGATCTGGAGAA',
 'CTCTCATCTCACAGAGGTGTTATCGCTGACAATCAAGCAAAATGGGCTGTCCCGACAACACGGACAGATGACAAGTTGCGAATGGAGACATGCTTCCAGC',
 'GTCAGGTTAATTGGAAACCCGGATGTGAGCGGGCCCAAACTAACAGGGGCACTAATAGGTATATTATCCTTATTTGTGGAGTCTCCAGGTCAATTGATTC',
 'CTAACACGGTATTACATCTTCACGTCGAAACAGATTGTTGCGTGATCCCGATG

In [10]:
def overlap(a, b, min_length=1):
    """Return length of longest suffix of 'a' matching prefix of 'b'."""
    start = 0
    while True:
        start = a.find(b[:min_length], start)
        if start == -1:
            return 0
        if b.startswith(a[start:]):
            return len(a) - start
        start += 1

def greedy_scs(reads, k=1):
    reads = reads[:]  # copy
    while len(reads) > 1:
        max_olen = -1
        best_a, best_b = None, None
        best_merged = None
        
        # 找到 overlap 最大的两个 reads
        for i in range(len(reads)):
            for j in range(len(reads)):
                if i == j:
                    continue
                a, b = reads[i], reads[j]
                olen = overlap(a, b, min_length=k)
                if olen > max_olen:
                    max_olen = olen
                    best_a, best_b = a, b
                    best_merged = a + b[olen:]
        
        # 合并
        reads.remove(best_a)
        reads.remove(best_b)
        reads.append(best_merged)
    
    return reads[0]


In [11]:
assembled = greedy_scs(sequences, k=12) 

In [12]:
len(assembled) 

15894

In [13]:
assembled.count('A')

4633

In [14]:
assembled.count('T')

3723